# TRACR Purdue Data Collection Demo

This notebook demonstrates METS-R controlled vehicles projected into CARLA for data collection on `PurdueWestLafayette_centered`. The demo surface combines the live METS-R Viz stream, a CARLA bird-eye tracking camera, an enlarged ego-centered C-V2X/BSM panel showing Kafka or Simu5G records as SAE J2735 core fields, and an enlarged CARLA LiDAR panel from the selected sensor vehicle.

## Implementation proposal

The demo runs in METS-R authoritative projection mode: METS-R owns all vehicle movement, CARLA mirrors those vehicle locations for sensor collection, and the configured C-V2X/BSM stream exposes V2X messages: Kafka by default, or Simu5G through the OMNeT++ bridge. The helper in `utils/cosim_support.py` keeps the notebook focused on presentation cells.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "dashboard_demos":
    REPO_ROOT = REPO_ROOT.parent
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import utils.cosim_support as cosim_support
cosim_support = importlib.reload(cosim_support)

TRACRDashboard = cosim_support.TRACRDashboard
launch_tracr_demo = cosim_support.launch_tracr_demo

print("Repository root:", REPO_ROOT)


Repository root: c:\Users\ALei\Documents\GitHub\METS-R_HPC


## Launch services

This starts METS-R in Docker, CARLA with the Purdue map, creates private trips, marks a subset as C-V2X BSM emitters, and starts the METS-R Viz WebSocket stream. The C-V2X/BSM panel is ego-centered: rows show the focused vehicle role plus SAE J2735 `coreData` fields (`msgCnt`, `secMark`, `lat`, `long`, `speed`, `heading`, etc.) and hide unrelated messages. Link latency/range are shown only as metadata because they are not BSM fields. It uses Kafka by default; set `bsm_stream_source="simu5g"` to route sender-to-ego BSMs through the Simu5G/OMNeT++ bridge on `127.0.0.1:9099`. Keep Docker Desktop host networking enabled before running this cell.

In [20]:
runtime = launch_tracr_demo(
    run_config="configs/run_cosim_CARLAPurdue.json",
    private_vehicle_count=50,
    v2x_vehicle_count=10,
    private_vehicle_start_id=1,
    start_kafka=None,  # auto: Kafka only when bsm_stream_source="kafka"
    bsm_stream_source="kafka",  # "kafka" or "simu5g"
    simu5g_host="127.0.0.1",
    simu5g_port=9099,
    start_metsr=True,
    start_carla=True,
    carla_camera_z=100.0,
    bsm_poll_timeout_ms=1,
    bsm_max_records=120,
    projection_heading_smoothing=0.35,
    random_seed="auto",
)

{"random_seed": runtime.random_seed, "viz_info": runtime.viz_info}

No registered METS-R client helper servers found.
Visualization server stopped.
No running METS-R Docker containers found.
Connection established!
METS-R Vis live stream is available at ws://127.0.0.1:8765; origin=(-86.0, 40.0); call render() to send frames.


{'random_seed': 438828552,
 'viz_info': {'host': '0.0.0.0',
  'port': 8765,
  'url': 'ws://127.0.0.1:8765',
  'local_url': 'ws://127.0.0.1:8765',
  'localhost_url': 'ws://localhost:8765',
  'browser_url': 'ws://127.0.0.1:8765',
  'localhost_reachable': False,
  'manifest': {'format': 'metsr-trajectory-binary',
   'version': 8,
   'byteOrder': 'bigEndian',
   'chunkMagic': 'MRTB',
   'coordScale': 100000,
   'initialX': -86.0,
   'initialY': 40.0,
   'tickInterval': 1,
   'linkSnapshotInterval': 1,
   'chunkTickLimit': 1,
   'chunks': [],
   'activeChunk': None,
   'roadIdDictionary': ['-104',
    '-12073',
    '-12074',
    '-12075',
    '-12078',
    '-12079',
    '-12080',
    '-12081',
    '-12083',
    '-12084',
    '-12085',
    '-12086',
    '-12088',
    '-12089',
    '-12102',
    '-12125',
    '-12141',
    '-12145',
    '-12146',
    '-12147',
    '-12148',
    '-12149',
    '-12150',
    '-12151',
    '-12152',
    '-12153',
    '-12154',
    '-12191',
    '-12192',
    '-12

## Demo dashboard

Run this cell to start a local browser dashboard. Open the printed `http://127.0.0.1:8899/index.html` URL in Chrome or Edge. The dashboard embeds the hosted Purdue METS-R Vis page and preloads it with `Map=12`, the live `StreamURL`, and the current ego `VehicleID` plus an inferred `VehicleType` (falling back to private EV type `1` until the live record is available), so you should not need to manually choose the map or paste the stream URL. Press `F11` for true full screen.

In [21]:
dashboard = TRACRDashboard(
    stream_url=runtime.viz_info["url"],
    fullscreen=True,
    metsr_viz_map=12,
    metsr_viz_vehicle_type=1,
    bsm_stream_label=runtime.bsm_stream_label,
    bsm_ego_only=True,
    lidar_min_update_interval_s=1.0,
)
dashboard_url = dashboard.display_external(port=8899)
dashboard_url

Serving c:\Users\ALei\Documents\GitHub\METS-R_HPC\output\tracr_dashboard with CORS enabled on port 8899...


'http://127.0.0.1:8899/index.html'

## Preview attacked-BSM highlighting

Run the next cell after creating the dashboard. It defines a synthetic attacked-BSM factory but does not send anything yet. The notebook live loop calls this factory after the Kafka or Simu5G polling phase, gives the fake record the same receiver shape as the normal batch, and includes it in every dashboard refresh. The red row therefore remains visible alongside the normal BSM rows without modifying either backend.

In [22]:
def build_fake_attack_bsm(runtime, step_result, normal_bsm_records):
    demo_v2x_ids = list(getattr(runtime, 'v2x_vehicle_ids', []) or [])
    if not demo_v2x_ids:
        return []

    projection = step_result.get('tracr_projection', {}) if isinstance(step_result, dict) else {}
    demo_ego_vehicle_id = (
        projection.get('focus_vehicle')
        or getattr(runtime, 'focus_vehicle_id', None)
        or getattr(getattr(runtime, 'sensor_panel', None), 'target_vehicle_id', None)
        or demo_v2x_ids[0]
    )
    demo_attack_sender_id = next(
        (vehicle_id for vehicle_id in demo_v2x_ids if str(vehicle_id) != str(demo_ego_vehicle_id)),
        f'attack-demo-{demo_ego_vehicle_id}',
    )
    tick = int(getattr(runtime.metsr, 'current_tick', 0) or 0)

    record = {
        'sender_id': demo_attack_sender_id,
        'vehicle_id': demo_attack_sender_id,
        'vid': demo_attack_sender_id,
        'attacked': True,
        'attack_id': f'tracr-dashboard-preview:{tick}',
        'attack_type': 'synthetic_preview',
        'message_name': 'BasicSafetyMessage',
        'message_standard': 'SAE J2735 BSMcoreData',
        'coreData': {
            'id': str(demo_attack_sender_id),
            'msgCnt': tick % 128,
            'secMark': tick % 60000,
            'lat': 404250000,
            'long': -869150000,
            'elev': 1900,
            'speed': 500,
            'heading': 14400,
            'brakes': {'wheelBrakes': 'unavailable'},
            'size': {'width': 200, 'length': 450},
        },
    }

    source = str(getattr(runtime, 'bsm_stream_source', '') or '').lower()
    has_ego_deliveries = any(
        str(cosim_support.bsm_receiver_id(row)) == str(demo_ego_vehicle_id)
        for row in normal_bsm_records
    )
    if source == 'simu5g' or has_ego_deliveries:
        record['_topic'] = 'v2x_rx_bsm'
        record['receiver_id'] = demo_ego_vehicle_id
        record['target_vehicle_id'] = demo_ego_vehicle_id
    else:
        record['_topic'] = 'bsm'

    return [record]


print('Configured one synthetic attacked BSM per live-loop iteration; run the next cell.')

Configured one synthetic attacked BSM per benchmark iteration; run the live-loop cell next.


## Run the live loop

Each iteration advances METS-R and CARLA one synchronized tick, mirrors METS-R private vehicles into CARLA, tracks the selected sensor vehicle in the CARLA bird-eye view, refreshes the latest LiDAR callbacks, filters the C-V2X/BSM panel to the current ego vehicle, and pushes frames to METS-R Viz. The demo loop intentionally throttles expensive side outputs: Kafka polling, dashboard redraws, and METS-R Viz rendering can run every N ticks while METS-R/CARLA synchronization still runs every tick.

In [23]:
# Run and profile the live loop directly in the notebook.
import time
import numpy as np

ticks = 2000
sleep_s = 0.0
render_wait_timeout = 0
render_every = 2
dashboard_every = 6
bsm_every = 30
sensor_every = 1

deps = cosim_support._deps()
profile_samples = []
last_result = None

for tick_index in range(ticks):
    timings = {'deps': 0.0}
    total_start = time.perf_counter()
    final_tick = tick_index == ticks - 1
    render_now = final_tick or tick_index % max(1, render_every) == 0
    dashboard_now = final_tick or tick_index % max(1, dashboard_every) == 0
    sensors_now = final_tick or tick_index % max(1, sensor_every) == 0
    poll_bsm_now = final_tick or tick_index % max(1, bsm_every) == 0

    start = time.perf_counter()
    if runtime.world is not None:
        runtime.world.tick()
    runtime.metsr.tick(wait_forever=True, poll_timeout=5)
    step_result = {
        'state': runtime.carla_state,
        'vehicles': [],
        'fleet_controlled': True,
    }
    timings['metsr_step'] = (time.perf_counter() - start) * 1000.0

    start = time.perf_counter()
    projection_info = cosim_support._sync_tracr_road_context_vehicles(runtime, deps)
    step_result['tracr_projection'] = projection_info
    timings['road_projection'] = (time.perf_counter() - start) * 1000.0

    start = time.perf_counter()
    cosim_support._keep_carla_projection_passive(runtime.carla_state, deps['carla'])
    timings['passive_carla'] = (time.perf_counter() - start) * 1000.0

    if sensors_now and runtime.sensor_panel is not None:
        start = time.perf_counter()
        preferred_vehicle_ids = []
        if projection_info.get('focus_vehicle') is not None:
            preferred_vehicle_ids.append(projection_info['focus_vehicle'])
        preferred_vehicle_ids.extend(getattr(runtime, 'v2x_vehicle_ids', []) or [])
        runtime.sensor_panel.ensure_sensors(
            runtime.carla_state,
            preferred_vehicle_ids=preferred_vehicle_ids,
        )
        timings['sensor_sync'] = (time.perf_counter() - start) * 1000.0
    else:
        timings['sensor_sync'] = 0.0

    start = time.perf_counter()
    bsm_stream = getattr(runtime, 'bsm_stream', None) or getattr(runtime, 'kafka_processor', None)
    normal_bsm_records = list(getattr(runtime, '_tracr_last_bsm_records', []) or [])
    if bsm_stream is not None and poll_bsm_now:
        timeout_ms = int(getattr(runtime, 'bsm_poll_timeout_ms', 1) or 1)
        max_records = int(getattr(runtime, 'bsm_max_records', 120) or 120)
        try:
            polled_records = bsm_stream.process_bsm(
                runtime=runtime,
                timeout_ms=timeout_ms,
                max_records=max_records,
            ) or []
        except TypeError:
            polled_records = bsm_stream.process_bsm(
                timeout_ms=timeout_ms,
                max_records=max_records,
            ) or []
        # Retain the last non-empty real batch between sparse backend polls.
        if polled_records:
            normal_bsm_records = list(polled_records)
            runtime._tracr_last_bsm_records = list(normal_bsm_records)
    if getattr(bsm_stream, 'last_error', ''):
        step_result['bsm_stream_error'] = getattr(bsm_stream, 'last_error', '')
    timings['bsm_stream'] = (time.perf_counter() - start) * 1000.0 if poll_bsm_now else 0.0

    start = time.perf_counter()
    bsm_records = list(normal_bsm_records)
    bsm_records.extend(build_fake_attack_bsm(runtime, step_result, normal_bsm_records))
    timings['bsm_injector'] = (time.perf_counter() - start) * 1000.0

    render_info = None
    render_error = None
    if render_now:
        start = time.perf_counter()
        try:
            render_info = runtime.metsr.render(client_wait_timeout=render_wait_timeout)
            ego_vehicle_id = cosim_support._runtime_ego_vehicle_id(runtime, step_result)
            if isinstance(render_info, dict):
                render_info['tracr_viz_selected_vehicle_id'] = ego_vehicle_id
                ego_vehicle_type = cosim_support._runtime_metsr_vis_vehicle_type(runtime, ego_vehicle_id)
                if ego_vehicle_type is not None:
                    render_info['tracr_viz_selected_vehicle_type'] = ego_vehicle_type
        except Exception as exc:
            render_error = str(exc).splitlines()[0]
        timings['metsr_viz_render'] = (time.perf_counter() - start) * 1000.0
    else:
        render_info = {'skipped': True}
        timings['metsr_viz_render'] = 0.0

    if dashboard_now:
        start = time.perf_counter()
        dashboard.update(
            runtime,
            step_result,
            bsm_records,
            render_info=render_info,
            render_error=render_error,
        )
        timings['dashboard_update'] = (time.perf_counter() - start) * 1000.0
    else:
        timings['dashboard_update'] = 0.0

    timings['total'] = (time.perf_counter() - total_start) * 1000.0
    profile_samples.append(timings)
    last_result = {
        'step_result': step_result,
        'bsm_records': bsm_records,
        'normal_bsm_records': normal_bsm_records,
        'render_info': render_info,
        'render_error': render_error,
        'profile_ms': timings,
    }
    if sleep_s:
        time.sleep(float(sleep_s))

profile_summary_ms = {}
for key in sorted({key for sample in profile_samples for key in sample}):
    values = np.asarray([sample[key] for sample in profile_samples if key in sample], dtype=float)
    if values.size:
        profile_summary_ms[key] = {
            'mean': float(values.mean()),
            'p50': float(np.percentile(values, 50)),
            'p95': float(np.percentile(values, 95)),
            'max': float(values.max()),
            'total': float(values.sum()),
        }

last_result['profile_samples'] = len(profile_samples)
last_result['profile_summary_ms'] = profile_summary_ms
last_result["profile_summary_ms"]


KeyboardInterrupt: 

## Cleanup

Run this when the demo is done. Set `stop_kafka=True` if this notebook started Kafka and no other notebook needs it.

In [6]:
runtime.close(stop_kafka=False)